In [ ]:
from __future__ import annotations
import os
import sys
import json
import glob
import time
import gc
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False

# --- Competition Physical Constants ---
# (Z, Y, X) physical resolution in microns (µm) per voxel
SCALE_ZYX = np.array([1.625, 0.40625, 0.40625])      

# --- Locate Dataset Directories ---
CANDIDATES = [
    "/kaggle/input/biohub-cell-tracking-during-development",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
    "data",
]
ROOT_DIR = next((p for p in CANDIDATES if Path(p, "test").exists()), "data")
TEST_DIR = Path(ROOT_DIR) / "test"
TRAIN_DIR = Path(ROOT_DIR) / "train"
OUTPUT_FILE = "submission.csv"

print(f"[STATUS] Data Root Verified: {ROOT_DIR}")
print(f"[STATUS] Total Test Movies Found: {len(list(TEST_DIR.glob('*.zarr')))}")

# --- High-Performance Pipeline Configuration ---
@dataclass
class PipelineConfig:
    scale: np.ndarray = field(default_factory=lambda: SCALE_ZYX.copy())
    
    # 1. Detection Phase Settings (Calibrated for Moderate Density to avoid FP node penalty)
    xy_downsample: int = 4            # Downsample anisotropic X/Y to match Z grid spacing (~isotropic)
    smooth_sigma: float = 1.0         # Gaussian kernel width for denoising
    thresh_rel: float = 0.18          # Threshold parameter to secure higher edge recall
    use_otsu: bool = True             # Auto-threshold baseline adaptively
    min_peak_dist: int = 2            # Minimum local pixel distance between cell centroids
    nms_radius_um: float = 4.0        # Physical exclusion radius in microns to avoid cell merging
    border_z: int = 0                 # Retain border frames (cells present close to frame boundaries)
    max_nodes_per_frame: int = 450    # Upper bound to safeguard host-metric node penalty limits
    refine_radius_z: int = 2          # Z window size for center-of-mass refinement
    refine_radius_yx: int = 5         # YX window size for center-of-mass refinement
    
    # 2. Association/Linking Phase Settings
    max_link_dist_um: float = 11.0    # Fallback relaxation gate (Anchor: 1.3x of P99 displacement)
    tight_gate_um: float = 7.0        # Pass-1 priority bounding gate (Anchor: 0.83x of P99 displacement)
    use_motion_prediction: bool = True # Tracks vectors dynamically to handle directed cell migration
    
    # 3. Mitosis / Mitotic Division Settings
    detect_divisions: bool = False    # Safe Default: Spares division Jaccard penalty from rare FPs
    div_parent_dist_um: float = 9.0   
    div_sister_dist_um: float = 9.0   
    div_min_count_gain: int = 1       
    
    # 4. Graph Refinement / Pruning
    prune_isolated: bool = True       # Immediately drop unlinked nodes to eliminate ghost detections
    keep_strong_isolated_q: float = 0.0

CFG = PipelineConfig()
print("[STATUS] Configuration parameters successfully loaded and scaled.")


In [ ]:
# --- Step 2: Stream Readers for OME-Zarr Volumes ---

def read_array_meta(zarr_path: Path | str) -> dict:
    """
    Parses structural parameters directly from the local zarr json array descriptor.
    Bypasses standard libraries to keep execution completely clean and lightweight.
    """
    meta_file = Path(zarr_path) / "0" / "zarr.json"
    with open(meta_file, "r") as f:
        meta_data = json.load(f)
        
    return {
        "shape": tuple(meta_data["shape"]), 
        "dtype": np.dtype(meta_data["data_type"]),
        "chunks": tuple(meta_data["chunk_grid"]["configuration"]["chunk_shape"])
    }

# Keep initialized zarr arrays cached to minimize repetitive I/O descriptor overhead
_ZARR_CACHE = {}

def load_volume(zarr_path: Path | str, t: int, meta: dict | None = None) -> np.ndarray:
    """
    Extracts a singular 3D volume slice at timepoint 't' with shape (Z, Y, X).
    Leverages native zarr reading, falling back directly to low-level blosc2
    decompression from system blocks if high-level packages are absent.
    """
    try:
        import zarr
        cache_key = str(zarr_path)
        if cache_key not in _ZARR_CACHE: 
            _ZARR_CACHE[cache_key] = zarr.open(cache_key, mode="r")["0"]
        return np.asarray(_ZARR_CACHE[cache_key][t])
    except Exception:
        # High-performance local fallback if standard zarr mounting fails offline [1]
        import blosc2
        if meta is None: 
            meta = read_array_meta(zarr_path)
            
        # Target the raw chunk cluster files directly following the zarr directory schema
        chunk_file = Path(zarr_path) / "0" / "c" / str(t) / "0" / "0" / "0"
        with open(chunk_file, "rb") as f:
            decompressed_buffer = blosc2.decompress(f.read())
            
        # Reconstruct structural 3D matrix using dimension array targets (Z, Y, X) [1]
        return np.frombuffer(decompressed_buffer, dtype=meta["dtype"]).reshape(meta["shape"][1:])

# Verify metadata read pipeline is fully ready for incoming loops
print("[STATUS] Chunk data parsers and zero-memory IO handlers ready.")


In [ ]:
# --- Step 3: Anisotropic 3D Centroid Extractor & Local Maxima Suppressor ---

def _block_mean_xy(vol: np.ndarray, factor: int) -> np.ndarray:
    """
    Downsamples the X and Y spatial grids by averaging blocks of size (factor x factor).
    This corrects the anisotropy so that Z-axis spacing closely matches XY spacing.
    """
    Z, Y, X = vol.shape
    Y_rounded, X_rounded = (Y // factor) * factor, (X // factor) * factor
    
    # Safe slice to matching block bounds without reallocation
    v = vol[:, :Y_rounded, :X_rounded].astype(np.float32, copy=False)
    return v.reshape(Z, Y_rounded // factor, factor, X_rounded // factor, factor).mean(axis=(2, 4))

def _robust_threshold(smoothed_grid: np.ndarray, cfg: PipelineConfig) -> float:
    """
    Computes an adaptive intensity cut-off floor. Combines percentile margins
    with an automated Otsu threshold filter to handle background intensity drift.
    """
    bg_floor = float(np.median(smoothed_grid))
    peak_intensity = float(np.percentile(smoothed_grid, 99.9))
    
    relative_floor = bg_floor + cfg.thresh_rel * max(peak_intensity - bg_floor, 1e-6)
    
    if cfg.use_otsu and HAS_SKIMAGE:
        try:
            return max(float(threshold_otsu(smoothed_grid)), relative_floor)
        except Exception:
            pass
    return max(float(np.percentile(smoothed_grid, 96.0)), relative_floor)

def _find_local_peaks(smoothed_grid: np.ndarray, threshold: float, min_dist: int) -> np.ndarray:
    """
    Discovers candidate cell peak targets. Uses scikit-image if available,
    otherwise falls back to a structural maximum_filter matrix transformation.
    """
    if HAS_SKIMAGE:
        return peak_local_max(
            smoothed_grid, 
            min_distance=int(min_dist), 
            threshold_abs=threshold, 
            exclude_border=False
        ).astype(np.int32)
        
    filter_size = 2 * int(min_dist) + 1
    max_filtered = maximum_filter(smoothed_grid, size=filter_size, mode="nearest")
    return np.argwhere((smoothed_grid >= max_filtered) & (smoothed_grid > threshold)).astype(np.int32)

def _refine_centroid_mass(vol: np.ndarray, zyx_coord: np.ndarray, cfg: PipelineConfig) -> tuple[np.ndarray, float]:
    """
    Performs high-resolution sub-pixel center-of-mass adjustment 
    on the native, un-pooled image volume for high coordinate precision.
    """
    Z, Y, X = vol.shape
    z, y, x = (int(round(v)) for v in zyx_coord)
    
    # Crop boundaries containment checks
    z0, z1 = max(0, z - cfg.refine_radius_z), min(Z, z + cfg.refine_radius_z + 1)
    y0, y1 = max(0, y - cfg.refine_radius_yx), min(Y, y + cfg.refine_radius_yx + 1)
    x0, x1 = max(0, x - cfg.refine_radius_yx), min(X, x + cfg.refine_radius_yx + 1)
    
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32)
    bg_floor = float(crop.min())
    
    weights = np.clip(crop - bg_floor, 0, None)
    weight_sum = float(weights.sum())
    
    if weight_sum <= 0:
        return np.array([z, y, x], dtype=float), 0.0
        
    grid_z, grid_y, grid_x = np.mgrid[z0:z1, y0:y1, x0:x1]
    refined_coords = np.array([
        (grid_z * weights).sum(),
        (grid_y * weights).sum(),
        (grid_x * weights).sum()
    ]) / weight_sum
    
    return refined_coords, float(crop.max() - bg_floor)

def _physical_nms(coords: np.ndarray, scores: np.ndarray, radius_um: float, scale: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Suppresses redundant close-proximity cell node detections using real spatial micron distances.
    Prevents crowding of overlapping predictions that would incur strict score penalties.
    """
    if len(coords) <= 1:
        return coords, scores
        
    # Scale coordinates into true physical micrometers (µm)
    physical_pts = coords * scale[None, :]
    descending_order = np.argsort(-scores)
    
    spatial_tree = cKDTree(physical_pts)
    eliminated_mask = np.zeros(len(coords), dtype=bool)
    retained_indices = []
    
    for idx in descending_order:
        if eliminated_mask[idx]:
            continue
        retained_indices.append(int(idx))
        # Mask out all nodes within the physical micrometer threshold radius
        neighbor_indices = spatial_tree.query_ball_point(physical_pts[idx], r=radius_um)
        eliminated_mask[neighbor_indices] = True
        
    retained_indices = np.array(retained_indices)
    return coords[retained_indices], scores[retained_indices]

def detect(vol: np.ndarray, cfg: PipelineConfig) -> tuple[np.ndarray, np.ndarray]:
    """
    Main detection workflow: processes 3D matrices, thresholds backgrounds,
    extracts node local maxima, refines center-of-mass, and enforces physical NMS.
    """
    downsample_factor = cfg.xy_downsample
    
    # 1. Spatial Downsampling & Smoothing
    pooled_volume = _block_mean_xy(vol, downsample_factor) if downsample_factor > 1 else vol.astype(np.float32)
    if cfg.smooth_sigma > 0:
        smoothed = gaussian_filter(pooled_volume, cfg.smooth_sigma)
    else:
        smoothed = pooled_volume
        
    # 2. Local Peak Mapping
    adaptive_floor = _robust_threshold(smoothed, cfg)
    peaks = _find_local_peaks(smoothed, adaptive_floor, cfg.min_peak_dist)
    
    if len(peaks) == 0:
        return np.zeros((0, 3)), np.zeros(0)
        
    # Re-scale pooled peak indices back to native grid space coordinates
    unpooled_peaks = peaks.astype(float)
    unpooled_peaks[:, 1] = unpooled_peaks[:, 1] * downsample_factor + (downsample_factor - 1) / 2
    unpooled_peaks[:, 2] = unpooled_peaks[:, 2] * downsample_factor + (downsample_factor - 1) / 2
    
    # 3. Sub-pixel Center-of-Mass Intensity Refinement
    Z_depth = vol.shape[0]
    validated_coords, point_scores = [], []
    
    for peak in unpooled_peaks:
        if peak[0] < cfg.border_z or peak[0] > (Z_depth - 1 - cfg.border_z):
            continue
        coord, score = _refine_centroid_mass(vol, peak, cfg)
        validated_coords.append(coord)
        point_scores.append(score)
        
    if not validated_coords:
        return np.zeros((0, 3)), np.zeros(0)
        
    coords_arr = np.array(validated_coords)
    scores_arr = np.array(point_scores)
    
    # 4. Filter Redundancies using True Micron Exclusion Space via NMS
    final_coords, final_scores = _physical_nms(coords_arr, scores_arr, cfg.nms_radius_um, cfg.scale)
    
    # 5. Cap Peak Counts if Threshold Limits are Breached
    if cfg.max_nodes_per_frame and len(final_coords) > cfg.max_nodes_per_frame:
        cutoff_indices = np.argsort(-final_scores)[:cfg.max_nodes_per_frame]
        final_coords = final_coords[cutoff_indices]
        final_scores = final_scores[cutoff_indices]
        
    return final_coords, final_scores

print("[STATUS] Anisotropic 3D Voxel cell detector compiled and ready.")


In [ ]:
# --- Step 4: Two-Pass Motion-Aware Hungarian Connector ---

def _link_nodes_hungarian(prev_xyz: np.ndarray, curr_xyz: np.ndarray, cfg: PipelineConfig, prev_velocity: np.ndarray | None = None) -> list[tuple[int, int]]:
    """
    Connects detected node coordinates between frames t and t+1 using an optimized
    two-pass Linear Sum Assignment (Hungarian) method based on true micron distances.
    """
    if len(prev_xyz) == 0 or len(curr_xyz) == 0:
        return []
        
    # Scale raw matrix positions directly to physical micrometer space
    prev_microns = prev_xyz * cfg.scale[None, :]
    curr_microns = curr_xyz * cfg.scale[None, :]
    
    # Apply velocity direction vector shift if historical motion trend is available
    if cfg.use_motion_prediction and prev_velocity is not None:
        predicted_positions = prev_microns + (0.5 * prev_velocity)
    else:
        predicted_positions = prev_microns
        
    num_prev, num_curr = len(prev_microns), len(curr_microns)
    BIG_COST = 1e9  # Penalty value for connections exceeding micron gates
    
    # Defensive Fallback: If population sizes explode, switch to greedy cKDTree matching
    if num_prev * num_curr > 4000000:
        spatial_tree = cKDTree(curr_microns)
        matches_pool = []
        for i, pt in enumerate(prev_microns):
            for j in spatial_tree.query_ball_point(pt, r=cfg.max_link_dist_um):
                dist = float(np.linalg.norm(pt - curr_microns[j]))
                matches_pool.append((dist, i, int(j)))
        matches_pool.sort()
        
        assigned_prev, assigned_curr, final_links = set(), set(), []
        for dist, i, j in matches_pool:
            if i in assigned_prev or j in assigned_curr:
                continue
            assigned_prev.add(i)
            assigned_curr.add(j)
            final_links.append((i, j))
        return final_links

    # Core Hungarian optimization routine wrapped inside a selective bounding gate
    def _run_assignment(p_indices: np.ndarray, c_indices: np.ndarray, active_gate: float) -> list[tuple[int, int]]:
        if len(p_indices) == 0 or len(c_indices) == 0:
            return []
            
        # Calculate real coordinate offset vs shifted forward target matrices
        raw_distances = np.sqrt(((prev_microns[p_indices][:, None] - curr_microns[c_indices][None]) ** 2).sum(2))
        predicted_distances = np.sqrt(((predicted_positions[p_indices][:, None] - curr_microns[c_indices][None]) ** 2).sum(2))
        
        # Penalize entries that are physically impossible based on our gate distance
        cost_matrix = np.where(raw_distances > active_gate, BIG_COST, predicted_distances)
        
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        return [(int(p_indices[r]), int(c_indices[c])) for r, c in zip(row_ind, col_ind) if cost_matrix[r, c] < BIG_COST]

    # --- Pass 1: High-confidence assignment using a tight physical gate ---
    tight_gate = min(cfg.tight_gate_um, cfg.max_link_dist_um)
    secure_links = _run_assignment(np.arange(num_prev), np.arange(num_curr), tight_gate)
    
    # Identify leftovers from the first strict round
    matched_prev = {p for p, _ in secure_links}
    matched_curr = {c for _, c in secure_links}
    
    unmatched_prev = np.array([i for i in range(num_prev) if i not in matched_prev], dtype=int)
    unmatched_curr = np.array([j for j in range(num_curr) if j not in matched_curr], dtype=int)
    
    # --- Pass 2: Fallback assignment using full relaxed physical gate ---
    relaxed_links = _run_assignment(unmatched_prev, unmatched_curr, cfg.max_link_dist_um)
    
    return secure_links + relaxed_links

def _detect_conservative_divisions(prev_xyz: np.ndarray, curr_xyz: np.ndarray, current_links: list[tuple[int, int]], cfg: PipelineConfig) -> list[tuple[int, int]]:
    """
    Identifies rare mitosis cell-split division operations using strict distance rules.
    If 'detect_divisions' is disabled, it safely returns an empty set to prevent false positives.
    """
    if not cfg.detect_divisions or len(curr_xyz) - len(prev_xyz) < cfg.div_min_count_gain:
        return []
        
    prev_microns = prev_xyz * cfg.scale[None, :]
    curr_microns = curr_xyz * cfg.scale[None, :]
    
    assigned_children = {c for _, c in current_links}
    parent_map = {c: p for p, c in current_links}
    
    unassigned_nodes = [j for j in range(len(curr_xyz)) if j not in assigned_children]
    if not unassigned_nodes:
        return []
        
    prev_spatial_tree = cKDTree(prev_microns)
    mitosis_links = []
    
    for child_b_idx in unassigned_nodes:
        dist_to_parent, parent_idx = prev_spatial_tree.query(curr_microns[child_b_idx], k=1)
        if dist_to_parent > cfg.div_parent_dist_um:
            continue
            
        # Discover if this parent node already has an assigned daughter cell from Pass-1/Pass-2
        existing_siblings = [c for c, p in parent_map.items() if p == parent_idx]
        if not existing_siblings:
            continue
            
        # Enforce close-proximity distance thresholds between sister cells (daughter lineages)
        closest_sibling = min(existing_siblings, key=lambda c: np.linalg.norm(curr_microns[c] - curr_microns[child_b_idx]))
        sibling_distance = np.linalg.norm(curr_microns[closest_sibling] - curr_microns[child_b_idx])
        
        if sibling_distance <= cfg.div_sister_dist_um:
            mitosis_links.append((int(parent_idx), int(child_b_idx)))
            
    return mitosis_links

print("[STATUS] Two-pass tracking and division matching module compiled.")


In [ ]:
# --- Step 5: FIXED Sequential Movie Tracking Engine ---

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

def track_whole_movie(load_vol_func, total_frames: int, dataset_name: str, cfg: PipelineConfig, t_max=0, verbose: bool = False) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Sequentially steps through a full 3D+time movie array block.
    FIXED: Explicitly forces the dynamic frame index 't' to update correctly.
    """
    node_data_list, edge_data_list = [], []
    node_confidence_scores = {}
    
    previous_frame_node_ids = []
    previous_frame_coords = np.zeros((0, 3))
    previous_velocity_vectors = None
    
    global_node_index = 1
    frame_node_counts = []
    total_division_events = 0
    
    if t_max > 0:
        total_frames = min(total_frames, t_max)
        
    # CRITICAL FIX: Loop strictly uses the current frame index 'current_t'
    for current_t in range(total_frames):
        # 1. Access the individual frame slice securely
        current_volume = load_vol_func(current_t)
        
        # 2. Extract cell positions
        detected_coords, detection_scores = detect(current_volume, cfg)
        
        # Immediate memory release
        del current_volume
        gc.collect()
        
        # 3. Register current frame nodes with unique tracking IDs
        current_frame_node_ids = list(range(global_node_index, global_node_index + len(detected_coords)))
        global_node_index += len(detected_coords)
        
        for local_idx, node_id in enumerate(current_frame_node_ids):
            c_zyx = detected_coords[local_idx]
            s_val = detection_scores[local_idx]
            
            # FIXED: We explicitly pass 'current_t' here, NOT a static value
            node_data_list.append((
                dataset_name, "node", node_id, int(current_t), 
                float(c_zyx[0]), float(c_zyx[1]), float(c_zyx[2]), -1, -1
            ))
            node_confidence_scores[node_id] = float(s_val)
            
        # 4. Perform tracking link assignment against previous timepoint frame data
        if current_t > 0 and len(previous_frame_node_ids) > 0:
            computed_links = _link_nodes_hungarian(previous_frame_coords, detected_coords, cfg, previous_velocity_vectors)
            mitosis_links = _detect_conservative_divisions(previous_frame_coords, detected_coords, computed_links, cfg)
            
            current_velocity_matrix = np.zeros((len(previous_frame_coords), 3))
            
            # Process standard tracking links (edges)
            for p_idx, c_idx in computed_links:
                edge_data_list.append((
                    dataset_name, "edge", -1, -1, -1.0, -1.0, -1.0, 
                    previous_frame_node_ids[p_idx], current_frame_node_ids[c_idx]
                ))
                current_velocity_matrix[p_idx] = (detected_coords[c_idx] - previous_frame_coords[p_idx]) * cfg.scale
                
            # Process division splits
            for p_idx, c_idx in mitosis_links:
                edge_data_list.append((
                    dataset_name, "edge", -1, -1, -1.0, -1.0, -1.0, 
                    previous_frame_node_ids[p_idx], current_frame_node_ids[c_idx]
                ))
                
            total_division_events += len({p_idx for p_idx, _ in mitosis_links})
            
            forward_velocity_map = np.zeros((len(detected_coords), 3))
            for p_idx, c_idx in computed_links:
                forward_velocity_map[c_idx] = current_velocity_matrix[p_idx]
            previous_velocity_vectors = forward_velocity_map
        else:
            previous_velocity_vectors = None
            
        # Save frame metrics and prepare variables for the next frame iteration
        previous_frame_node_ids = current_frame_node_ids
        previous_frame_coords = detected_coords
        frame_node_counts.append(len(detected_coords))
        
        if verbose and ((current_t + 1) % 20 == 0 or current_t == total_frames - 1):
            print(f"  -> [{dataset_name}] Processed frame {current_t+1}/{total_frames} | Live cells: {len(detected_coords)}")
            
    nodes_df = pd.DataFrame(node_data_list, columns=SUBMISSION_COLUMNS)
    edges_df = pd.DataFrame(edge_data_list, columns=SUBMISSION_COLUMNS)
    
    # 5. Clean up isolated / unconnected nodes
    if cfg.prune_isolated and len(edges_df) > 0:
        active_edge_node_ids = set(edges_df.source_id) | set(edges_df.target_id)
        
        if cfg.keep_strong_isolated_q > 0 and node_confidence_scores:
            score_cutoff = np.quantile(list(node_confidence_scores.values()), cfg.keep_strong_isolated_q)
            active_edge_node_ids |= {nid for nid, score in node_confidence_scores.items() if score >= score_cutoff}
            
        nodes_df = nodes_df[nodes_df.node_id.isin(active_edge_node_ids)].reset_index(drop=True)
        
    summary_stats = {
        "dataset": dataset_name,
        "n_nodes": len(nodes_df),
        "n_edges": len(edges_df),
        "cells_per_frame": float(np.mean(frame_node_counts)) if frame_node_counts else 0.0,
        "n_div": total_division_events,
        "T": total_frames
    }
    
    return nodes_df, edges_df, summary_stats

print("[STATUS] Fixed sequential movie tracking pipeline is fully operational.")


In [ ]:
# --- Step 6: Submission Generator and Test Set Execution Loop ---

def check_available_frames(zarr_path: Path) -> tuple[int, dict]:
    """
    Checks the exact frame structure details present inside the Zarr archive arrays.
    """
    meta_info = read_array_meta(zarr_path)
    total_frames = meta_info["shape"][0]
    
    # Verify file structures safely
    valid_frames = [
        t for t in range(total_frames) 
        if (zarr_path / "0" / "c" / str(t) / "0" / "0" / "0").exists()
    ]
    
    actual_t_count = max(valid_frames) + 1 if valid_frames else 0
    return actual_t_count, meta_info

def run_submission_pipeline(cfg: PipelineConfig = CFG):
    """
    Loops through the entire test target directory, runs the movie aggregation pipeline,
    and builds the final verified Kaggle submission dataset array.
    """
    submission_parts = []
    execution_stats = []
    
    # Grab all .zarr arrays inside the verified test set directory path
    test_movie_paths = sorted(TEST_DIR.glob("*.zarr"))
    
    for path in test_movie_paths:
        dataset_id = path.name.replace(".zarr", "")
        
        if not (path / "0" / "zarr.json").exists():
            print(f"  [WARNING] Bypassing target {dataset_id}: metadata json missing.")
            continue
            
        t_frames, metadata = check_available_frames(path)
        if t_frames == 0:
            print(f"  [WARNING] Bypassing target {dataset_id}: no frames found.")
            continue
            
        start_time = time.time()
        print(f"[PROCESS] Tracking movie dataset: {dataset_id} (Frames={t_frames})...")
        
        # Invoke the complete frame aggregators using an isolated lambda wrapper
        nodes_df, edges_df, stats = track_whole_movie(
            lambda t: load_volume(path, t, metadata), 
            t_frames, 
            dataset_id, 
            cfg,
            verbose=True
        )
        
        elapsed_sec = round(time.time() - start_time, 1)
        stats["sec"] = elapsed_sec
        execution_stats.append(stats)
        
        # Cache outputs for batch structural concatenation
        submission_parts.extend([nodes_df, edges_df])
        
        print(f"  -> Finished {dataset_id} in {elapsed_sec}s | Nodes: {stats['n_nodes']} | Edges: {stats['n_edges']}")
        
    # Standard schema assembly loop
    final_submission = pd.concat(submission_parts, ignore_index=True)
    final_submission.index.name = "id"
    
    return final_submission, pd.DataFrame(execution_stats)

# --- Step 6: Regenerate Corrected submission.csv ---

print("[START] Re-running cell tracking loop to fix sequential timeframes...")
submission_df, benchmark_stats = run_submission_pipeline(CFG)

# Write the final fixed file to disk
submission_df.to_csv(OUTPUT_FILE)
print(f"\n[SUCCESS] Final corrected output saved to '{OUTPUT_FILE}'.")

# --- Verification Pass ---
print("\n[VERIFICATION] Checking if time points 't' are updating properly:")
unique_t_values = submission_df["t"].unique()
print(f"Unique 't' values found in file: {unique_t_values}")

# Check first rows to see layout
print("\nFirst 5 rows of regenerated file:")
display(submission_df.head(5))



In [ ]:
# --- Final Absolute Integer Coordinate Fix ---

import pandas as pd
import numpy as np

# Load the file we generated
df = pd.read_csv("submission.csv")

print("[PROCESS] Converting floating coordinates to integers as per strict requirements...")

# Convert coordinates to integers only for 'node' rows
node_mask = df["row_type"] == "node"

# Round coordinates to nearest integer values safely
df.loc[node_mask, "z"] = df.loc[node_mask, "z"].round().astype(int)
df.loc[node_mask, "y"] = df.loc[node_mask, "y"].round().astype(int)
df.loc[node_mask, "x"] = df.loc[node_mask, "x"].round().astype(int)

# Ensure edge coordinates are strictly preserved as -1
edge_mask = df["row_type"] == "edge"
df.loc[edge_mask, ["z", "y", "x"]] = -1

# Save back the final integer submission file
df.to_csv("submission.csv", index=False)
print("[SUCCESS] Final submission.csv file updated with pure integer voxel coordinates!")

# --- Quick Preview Verification ---
df_preview = pd.read_csv("submission.csv")
print("\nVerifying final structure with pure integers:")
print(df_preview.head(5))


In [ ]:

print(os.path.exists("submission.csv"))
df_check = pd.read_csv("submission.csv", nrows=5)
print(df_check)
